In [ ]:
import pandas as pd

In [ ]:
df2_cleaned = pd.read_csv("../../data/emotion_preprocessed.csv")
df2_cleaned.dropna(inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

# Split text and labels first to avoid fitting the vectorizer on test data.
X = df2_cleaned["clean_title"]
y = df2_cleaned["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on training text only, then transform test text.
vectorizer = TfidfVectorizer(
    max_features=10000,
    # stop_words='english',
    # min_df=0.001, max_df=0.999
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
print(type(X_train), X_train.shape, X_test.shape)

BOW_train = pd.DataFrame.sparse.from_spmatrix(
    X_train,
    index=y_train.index,
    columns=vectorizer.get_feature_names_out(),
)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns

import matplotlib.pyplot as plt

# Predictions for confusion matrix and ROC
y_pred = log_reg.predict(X_test)
y_score = log_reg.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC curve
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

X_sample = X_train[np.random.choice(X_train.shape[0], 20000, replace=False)]

In [ ]:
import shap

explainer = shap.Explainer(log_reg, X_sample)
shap_values = explainer(X_sample)

In [ ]:
feature_names = vectorizer.get_feature_names_out().tolist()
shap_values.feature_names = feature_names
shap.plots.bar(shap_values[0], max_display=20)

In [ ]:
word_importance = dict(zip(feature_names, shap_values.values[0]))

In [ ]:
import joblib

joblib.dump(word_importance, "../models/shap_word_importance.pkl")

In [ ]:
# sorted word_importance par importance
word_importance_sorted = dict(sorted(word_importance.items(), key=lambda item: abs(item[1]), reverse=True))
word_importance_sorted

In [ ]:
import re


def color_text_full(text, word_importance, vectorizer, threshold=0.005):
    analyzer = vectorizer.build_analyzer()

    words = re.findall(r"\w+|\W+", text)  # garde mots + ponctuation
    colored_words = []

    for word in words:
        clean_tokens = analyzer(word)

        if len(clean_tokens) > 0:
            token = clean_tokens[0]
            importance = word_importance.get(token, 0)
        else:
            importance = 0

        if importance > threshold:
            color = "green"
        elif importance < -threshold:
            color = "red"
        else:
            color = "white"

        colored_word = f'<span style="color:{color}">{word}</span>'
        colored_words.append(colored_word)

    return "".join(colored_words)

In [ ]:
from IPython.display import display, HTML

html = color_text_full("I am depressed today but I am still a crush", word_importance, vectorizer)
display(HTML(html))

In [ ]:
word_importance_sorted

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Keep token names in SHAP plots instead of generic feature indices.
feature_names = vectorizer.get_feature_names_out().tolist()
shap_values.feature_names = feature_names

# Sample rows only for expensive plotting operations.
PLOT_SAMPLE_SIZE = 2000
rng_plot = np.random.default_rng(42)
if shap_values.values.shape[0] > PLOT_SAMPLE_SIZE:
    plot_idx = np.sort(rng_plot.choice(shap_values.values.shape[0], size=PLOT_SAMPLE_SIZE, replace=False))
else:
    plot_idx = np.arange(shap_values.values.shape[0])

shap_values_plot = shap_values[plot_idx]
X_sample_plot = X_sample[plot_idx]

# Cohesive visual style
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.facecolor": "#F8FAFC",
        "axes.facecolor": "#F8FAFC",
        "axes.edgecolor": "#CBD5E1",
        "axes.titleweight": "bold",
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "font.size": 11,
    }
)

# 1) Global importance on the plotting subset.
plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values_plot, max_display=15, show=False)
plt.title("Top 15 mots les plus influents (importance globale)")
plt.xlabel("Impact moyen absolu sur la prediction")
plt.tight_layout()
plt.show()

# 2) Feature impact distribution on the plotting subset.
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_plot, max_display=15, show=False)
plt.title("Distribution des contributions SHAP")
plt.tight_layout()
plt.show()

# 3) Local explanation for the post with the highest predicted depression score.
sample_idx = int(np.argmax(log_reg.predict_proba(X_sample_plot)[:, 1]))
global_idx = int(plot_idx[sample_idx])
plt.figure(figsize=(11, 6))
shap.plots.waterfall(shap_values[global_idx], max_display=12, show=False)
plt.title(f"Explication locale - echantillon {global_idx}")
plt.tight_layout()
plt.show()

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))
from src.dashboard.shap import get_shap_values, shap_graph

shap_values = get_shap_values(log_reg, X_train, sample_size=10000)
shap_graph(log_reg, X_train, vectorizer=vectorizer)